<div class="alert alert-block alert-success">

# Molecular simulations for drug discovery and design

### A Lorentz Center – CECAM workshop · Bogotá, Colombia · 23–27 November 2026

## Hands-on session: Ligand-Based Methods

</div>

**Ligand-based methods** use information about known ligands to make predictions about new compounds. Instead of relying on the structure of the protein target, they analyse the molecules themselves (their structure, fingerprints, physicochemical descriptors and measured activities) to infer what makes a compound active, soluble or toxic.

In this notebook, we will:

1. Represent molecules with RDKit and compute molecular descriptors
2. Explore chemical space with Principal Component Analysis (PCA)
3. Predict a molecular property (water solubility) with a Random Forest model
4. Improve the model with better descriptors and settings

> 💡 This session shows the full workflow of a ligand-based model: from structures, to numbers, to a map of chemical space, to a predictive model.

<div class="alert alert-block alert-warning">

**Running on Google Colab**

1. Go to **File → Save a copy in Drive**, so that you work on your own copy.
2. Run the first code cell: it installs RDKit (about 30 seconds). All other packages are already available in Colab.
3. If Colab disconnects (for example after a long pause), your code is kept but the variables are lost: use **Runtime → Run all** to get back to where you were.
</div>

<div class="alert alert-block alert-info">

**How this session works**

- Fill in the code where you see `### START MODIFICATION ###` or `### ADD YOUR CODE HERE ###`.
- Answer the questions in the **blue boxes** and discuss them.
- Run the cells **in order**. If something breaks, restart the runtime and run all cells from the top.
</div>

In [ ]:
# Install RDKit if it is not available
try:
    import rdkit
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
    import rdkit
print("RDKit version:", rdkit.__version__)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

<h2 style="color:green;"> Step 1: Basic cheminformatics with RDKit </h2>

We will use **RDKit**, one of the most widely used open-source cheminformatics libraries. It provides tools to represent, manipulate and analyse molecular structures in Python.

We will turn chemical structures, stored as **SMILES** strings, into numbers: molecular descriptors that can be plotted and used as input for machine learning models.

<h3 style="color:green;"> Step 1.1: Load a small dataset </h3>

We use a small subset of the **ESOL** dataset (Delaney, *J. Chem. Inf. Comput. Sci.* 2004, [doi:10.1021/ci034243x](https://pubs.acs.org/doi/10.1021/ci034243x)), which contains experimental water solubilities expressed as $\log S$ ($S$ in mol/L).

Higher $\log S$ means more soluble; lower (more negative) values mean poorly soluble. Poor solubility can limit the absorption, and therefore the bioavailability, of a drug.

In [ ]:
# Compound names, SMILES and experimental solubility (logS) from the ESOL dataset
data = {"Compound ID": ["1-Butene", "Ethanol", "Butane", "Butanethiol", "Benzene", "Pyridazine", "Pyridine", "Pyrimidine",
                        "2-Iodopropane", "Dipropyl ether", "1,2-Dichloroethane", "1-Pentene", "2-Hydroxypyridine", "Acetamide",
                        "Fluorobenzene", "Anisole", "Bromochloromethane", "Diethyl ether", "Ethane", "2-pyrrolidone",
                        "Diphenylmethane", "Ethyl benzoate", "Ethylbenzene", "Glycerol", "Isobutyl formate", "Isopropyl formate",
                        "Methanol", "Methyl benzoate", "Nitrobenzene", "Urea", "1,3-Butadiene", "1,3-Dichloropropane",
                        "1-Butanol", "1-Chloropropane", "1-Iodopropane", "1-Octanol", "1-Pentene", "1-Pentanol",
                        "2,2-Dimethylpropanol", "2,4-Dimethyl-3-pentanol", "2-Bromopropane", "2-Butanone", "2-Chloropropane",
                        "2-Hexanol", "2-Methyl-1-Butene", "2-Methylbutanol", "2-Nonanol", "2-Octanone", "2-Octanol", "2-Pentanone"],
        "SMILES": ["CCC=C", "CCO", "CCCC", "CCCCS", "c1ccccc1", "c1ccnnc1", "c1ccncc1", "c1cncnc1", "CC(C)I", "CCCOCCC",
                   "ClCCCl", "CCCC=C", "Oc1ccccn1", "CC(=O)N", "Fc1ccccc1", "COc1ccccc1", "ClCBr", "CCOCC", "CC", "O=C1CCCN1",
                   "C(c1ccccc1)c2ccccc2", "CCOC(=O)c1ccccc1", "CCc1ccccc1", "OCC(O)CO", "CC(C)COC=O", "CC(C)OC=O",
                   "CO", "COC(=O)c1ccccc1", "O=N(=O)c1ccccc1", "NC(=O)N", "C=CC=C", "ClCCCCl", "CCCCO", "CCCCl", "CCCI",
                   "CCCCCCCCO", "CCCC=C", "CCCCCO", "CC(C)(C)CO", "CC(C)C(O)C(C)C", "CC(C)Br", "CCC(=O)C", "CC(C)Cl",
                   "CCCCC(C)O", "CCC(=C)C", "CCC(C)CO", "CCCCCCCC(C)O", "CCCCCCC(=O)C", "CCCCCCC(C)O", "CCCC(=O)C"],
        "logS_measured": [-1.94, 1.1, -2.57, -2.18, -1.64, 1.1, 0.76, 1.1, -2.09, -1.62, -1.06, -2.68, 1.02, 1.58, -1.8, -1.85, -0.89,
                          -0.09, -1.36, 1.07, -4.08, -2.32, -2.77, 1.12, -1.01, -0.63, 1.57, -1.85, -1.8, 0.96, -1.87, -1.62, 0,
                          -1.47, -2.29, -2.39, -2.68, -0.6, -0.4, -1.22, -1.59, 0.52, -1.41, -0.89, -2.73, -0.47, -2.74, -2.05, -2.09, -0.19]}

df = pd.DataFrame(data)
print(f"{len(df)} molecules")
df.head(10)

Before doing anything with a dataset, **always check its quality**. A common problem is duplicates: the same molecule entered twice. If one copy ends up in the training set and the other in the test set, the model is tested on a molecule it has already seen, and its performance looks better than it really is.

Find the duplicated SMILES in the dataset (hint: use `df.duplicated(...)` with `keep=False`, to show all copies).

In [ ]:
### ADD YOUR CODE HERE ###
duplicates = 
duplicates

In [ ]:
# Remove the duplicates, keeping the first copy of each molecule
df = df.drop_duplicates("SMILES").reset_index(drop=True)
print(f"{len(df)} unique molecules")

<h3 style="color:green;"> Step 1.2: From SMILES to molecules </h3>

RDKit converts each SMILES string into a molecule object, which we can draw and analyse.

In [ ]:
# Convert SMILES to RDKit molecules and draw the first 12
mols = [Chem.MolFromSmiles(s) for s in df["SMILES"]]
Draw.MolsToGridImage(mols[:12], molsPerRow=6, subImgSize=(180, 140),
                     legends=list(df["Compound ID"][:12]))

<h3 style="color:green;"> Step 1.3: Compute molecular descriptors </h3>

**Descriptors** summarise important properties of a molecule as numbers. They are widely used in cheminformatics and QSAR (quantitative structure–activity relationship) models. We start with five simple but informative ones:

| Descriptor | Meaning |
|---|---|
| **MW** | Molecular weight |
| **LogP** | Predicted octanol/water partition coefficient: a measure of hydrophobicity |
| **HBD** | Number of hydrogen-bond donors |
| **HBA** | Number of hydrogen-bond acceptors |
| **TPSA** | Topological polar surface area: a measure of polarity |

In [ ]:
def compute_descriptors(smiles):
    ''' Compute a few physicochemical descriptors from a SMILES string. '''
    mol = Chem.MolFromSmiles(smiles)
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
    }

# Compute the descriptors for each compound
X = df["SMILES"].apply(compute_descriptors).apply(pd.Series)
X.head(10)

We obtained the **feature matrix** $X$: each row is a molecule, each column a descriptor. This is the input format for both PCA and machine learning.

Let's look at the range of each descriptor.

In [ ]:
X.describe().loc[["mean", "std", "min", "max"]].round(2)

<div class="alert alert-block alert-info">

**Question 2.** Compare the standard deviations of the descriptors. Which descriptors vary the most, in absolute numbers? Why could this be a problem for a method like PCA, which looks for the directions of largest variance?
</div>

<h2 style="color:green;"> Step 2: PCA to explore chemical space </h2>

Each molecule is now described by several numerical descriptors (MW, logP, HBD, HBA, TPSA). In real drug discovery projects, molecules are described by hundreds to thousands of numbers (fingerprints, fragments, 3D descriptors...).

This means each molecule corresponds to a point in a high-dimensional descriptor space: in our simple case 5D, but in real drug discovery cases also 100D-1000D.
We cannot visualise such spaces directly, so we need a way to reduce the dimensionality while keeping most of the information.<br><br>

**Principal Component Analysis (PCA)** is a linear dimensionality reduction technique use to visualize and explore data. PCA finds new axes, the **principal components** (PC1, PC2, ...), along which the data varies the most. PC1 captures the largest variance, PC2 the next largest, and so on. Plotting the molecules on PC1 and PC2 gives a 2D **map of chemical space**, often used to:

- see the chemical diversity of a dataset
- spot outliers
- compare chemical series
- check whether a training set covers the molecules we want to predict

### Standardise first

To give all descriptors the same weight, we first **standardise** them: subtract the mean and divide by the standard deviation of each column, so that each has mean 0 and standard deviation 1.

In [ ]:
# Standardise the descriptors (mean 0, standard deviation 1)
X_scaled = StandardScaler().fit_transform(X)

# PCA with 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Fraction of the variance explained by PC1 and PC2:", pca.explained_variance_ratio_.round(2))
print(f"Total: {pca.explained_variance_ratio_.sum():.0%}")

In [ ]:
# Map of chemical space, coloured by solubility
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df["logS_measured"], cmap="viridis", s=60, edgecolor="k")
for i, name in enumerate(df["Compound ID"]):
    ax.annotate(name, (X_pca[i, 0], X_pca[i, 1]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)")
ax.set_title("PCA of molecular descriptors")
plt.colorbar(sc, label="logS (measured)")
plt.show()

The principal components are combinations of the original descriptors. The **loadings** tell us how much each descriptor contributes to each component, which helps us interpret the map.

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=X.columns, columns=["PC1", "PC2"])
loadings.round(2)

<div class="alert alert-block alert-info">

**Question 3.**

1. Use the loadings: which descriptors dominate PC1 and PC2? Can you give each axis a rough chemical meaning?

2. Are some molecules isolated from the rest? What makes them different?

3. Do soluble and insoluble molecules occupy different regions of the map? Do you see a trend of logS along one of the axes?
</div>

<h2 style="color:green;"> Step 3: Predicting solubility with machine learning </h2>

Now we learn the relationship between the descriptors and the measured solubility with a **Random Forest regressor**: an ensemble of decision trees, each trained on a random subset of the data and of the descriptors.

The steps are:
1. Split the dataset into a training set and a test set
2. Train the model on the training set
3. Predict the test set and compare with the measured values
4. Assess the performance with metrics
5. Inspect which descriptors the model relies on

<h3 style="color:green;"> Step 3.1: Training and test sets </h3>

A model must be evaluated on molecules it has **not** seen during training.

- Before training any machine learning model, we need to separate the data we will use to learn patterns (training set) from the data we will use to evaluate the model (test set). If we evaluate the model on the same data it was trained on, it might appear to perform perfectly, but this can be misleading because the model could just memorize the training examples rather than learning general rules.

- Testing the model on data that is not seen during training prevents overfitting, making sure the ML model is really learning patterns.

- Typically, 70–80% of the data is used for training, and 20–30% for testing. 

In [ ]:
# Target of the model: the measured solubility
y = df["logS_measured"]

# Train/test split: 70% training, 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
print(f"Training set: {len(X_train)} molecules, test set: {len(X_test)} molecules")

<h3 style="color:green;"> Step 3.2: Train the model </h3>


- The Random Forest algorithm builds an ensemble of decision trees. Each tree is trained on a random subset of the data and descriptors.

- By averaging the predictions of many trees, Random Forest reduces errors and is less sensitive to noise or outliers.

- During training, the model learns relationships between descriptors (MW, LogP, HBD, HBA, TPSA) and the target property (logS).


Complete the cell below: create a `RandomForestRegressor` with 100 trees (`n_estimators=100`) and `random_state=0`, then train it on the training set with `.fit(...)`.

In [ ]:
### START MODIFICATION ###
model =
model.fit()
### END MODIFICATION ###

<h3 style="color:green;"> Step 3.3: Predict the test set </h3>

Once trained, the model is used to predict $logS$ for molecules it has not seen before (the test set).

The prediction is compared with experimental values allows us to check how well the model generalizes.

In [ ]:
y_pred = model.predict(X_test)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred, color="tab:blue", s=60, edgecolor="k")
for name, yt, yp in zip(df.loc[y_test.index, "Compound ID"], y_test, y_pred):
    ax.annotate(name, (yt, yp), fontsize=8, xytext=(4, 4), textcoords="offset points")
lims = [min(y.min(), y_pred.min()) - 0.3, max(y.max(), y_pred.max()) + 0.3]
ax.plot(lims, lims, "r--", label="perfect prediction")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Measured logS")
ax.set_ylabel("Predicted logS")
ax.set_title("Random Forest: test set")
ax.legend()
plt.show()

<h3 style="color:green;"> Step 3.4: Assess the performance </h3>

- The **root mean squared error (RMSE)** is the typical size of the prediction error, in logS units. Lower is better.
- The **coefficient of determination ($R^2$)** is the fraction of the variance of the measured values explained by the model: 1 is perfect, 0 is no better than always predicting the average, and negative is worse than that.

For reference, the experimental error of solubility measurements is often around 0.5 log units.

In [ ]:
def report(name, y_true, y_hat):
    rmse = np.sqrt(mean_squared_error(y_true, y_hat))
    print(f"{name:>28s}:  RMSE = {rmse:.2f} log units,  R² = {r2_score(y_true, y_hat):.2f}")

report("Random Forest, training set", y_train, model.predict(X_train))
report("Random Forest, test set", y_test, y_pred)

<h3 style="color:green;"> Step 3.5: Which descriptors matter? </h3>

A Random Forest reports how much each descriptor contributed to its predictions (**feature importance**). 

This can provide chemical insight: for example, if LogP or TPSA is highly important, it suggests solubility is strongly influenced by hydrophobicity or polarity.

Feature importance is a useful way to interpret “black-box” machine learning models in a chemistry context.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importances.plot.barh(color="tab:blue", figsize=(6, 3.5))
plt.xlabel("Feature importance")
plt.title("Random Forest feature importance")
plt.tight_layout()
plt.show()

<div class="alert alert-block alert-info">

**Question 4.**

1. Compare the performance on the training and test sets. Which error do you expect to be smaller?

2. Which descriptors are the most important? Does this make chemical sense?

3. Which test molecules are predicted worst? Look at their descriptors and at the training set: can you explain why?

<h2 style="color:green;"> Step 4: Improving the model </h2>

<h3 style="color:green;"> Step 4.1: More descriptors </h3>

Delaney's ESOL model predicts solubility from four descriptors: LogP, molecular weight, the number of **rotatable bonds**, and the **aromatic proportion** (the fraction of heavy atoms that are aromatic).

Complete the function below to add these two descriptors (hints: `Descriptors.NumRotatableBonds(mol)`; for the aromatic proportion, count the atoms for which `atom.GetIsAromatic()` is `True` in `mol.GetAtoms()`, and divide by `mol.GetNumHeavyAtoms()`).

In [ ]:
def compute_descriptors_extended(smiles):
    mol = Chem.MolFromSmiles(smiles)
    descriptors = compute_descriptors(smiles)
    ### START MODIFICATION ###
    descriptors["RotB"] = 
    descriptors["AromProp"] = 
    ### END MODIFICATION ###
    return descriptors

X_ext = df["SMILES"].apply(compute_descriptors_extended).apply(pd.Series)
X_ext.head()

In [ ]:
# Same split as before (same random_state), now with the extended descriptors
X_train_ext, X_test_ext, y_train, y_test = train_test_split(X_ext, y, test_size=0.30, random_state=0)

model_ext = RandomForestRegressor(n_estimators=100, random_state=0).fit(X_train_ext, y_train)
report("Random Forest, 5 descriptors", y_test, y_pred)
report("Random Forest, 7 descriptors", y_test, model_ext.predict(X_test_ext))

pd.Series(model_ext.feature_importances_, index=X_ext.columns).sort_values().plot.barh(color="tab:orange", figsize=(6, 4))
plt.xlabel("Feature importance")
plt.title("Feature importance with the extended descriptors")
plt.tight_layout()
plt.show()

<h3 style="color:green;"> Step 4.2: Hyperparameters </h3>

**Hyperparameters** are settings of the model that are chosen by us, not learned from the data. For a Random Forest, the most important are the number of trees (`n_estimators`) and the maximum depth of each tree (`max_depth`; `None` means no limit).

In [ ]:
for n_trees in [10, 50, 200]:
    for depth in [2, 4, None]:
        rf = RandomForestRegressor(n_estimators=n_trees, max_depth=depth, random_state=0).fit(X_train_ext, y_train)
        report(f"{n_trees} trees, max_depth={depth}", y_test, rf.predict(X_test_ext))

<div class="alert alert-block alert-info">

**Question 5.**

1. Did the two new descriptors improve the model? Which descriptors are now the most important?

2. How do the number of trees and the maximum depth affect the test performance?

3. We compared many settings on the same test set and could now pick the best one. Why is this a problem, and how should it be done instead?
</div>

<div class="alert alert-block alert-warning">

**Take-home messages**

- Ligand-based models turn molecules into numbers (descriptors, fingerprints) and learn from known data.
- Check the data first: duplicates, units and errors directly affect the model.
- Scale descriptors before PCA; PCA maps chemical space, but does not create clusters.
- Always evaluate on molecules the model has not seen, and compare with a simple baseline.
- A model is only reliable for molecules similar to its training set (its **applicability domain**).

</div>